In [42]:
import numpy as np
import pandas as pd


In [43]:
movies_ds = pd.read_csv("data/tmdb_5000_movies.csv")
credits_ds = pd.read_csv("data/tmdb_5000_credits.csv", low_memory = False)
# we see there are a bunch of extra commas in csv being treated as columns. let's get rid of these
credits_ds = credits_ds[["movie_id", "title", "cast", "crew"]]
df = movies_ds.merge(credits_ds, on = "title")
df_csv = df.to_csv("data/movies.csv", index = False)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [44]:
print(f"Dataset shape: {df.shape}")
print(f"First 5 rows:\n{df.head(5)}")

Dataset shape: (1492, 23)
First 5 rows:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...  

In [45]:
import ast
# note that some columns are just a list of dictionaries, where the relevant data value is just the value
# where the key is "name". The number of columns are still correct, as the list items are also separated by commas
# genres is a list of dictionaries, but there are multiple so we must extract the name value at each dictionary
# this is for multiple columns, so let's create a function that does this for us
def clean_list_of_dics(column):
    list_of_dics_as_series = df[column]
    print(list_of_dics_as_series.dtype) # shows us that the list of dics is being stored as a string
    # so now we need to convert it into the expected actual list of dics, using ast.literal_eval()
    new_series = []
    for list_of_dics_str in list_of_dics_as_series:
        try:
            list_of_dics = ast.literal_eval(list_of_dics_str)
            values = []
            for index in range(0, len(list_of_dics)):
                dic = list_of_dics[index]
                value = dic["name"]
                values.append(value)
            new_series.append(values)
        except Exception as exc:
            new_series.append([])
            row_num = df.index[df[column] == list_of_dics_str]
            print(f"Error at row {row_num} for column {column} : {exc}")
    df[column] = new_series
    cols_to_clean = ["genres", "keywords", "production_companies", "production_countries", "spoken_languages",
                     "cast", "crew"] # for cast, crew the only information really needed is their name
for col in cols_to_clean:
    clean_list_of_dics(col)


str
str
str
str
str
str
Error at row RangeIndex(start=218, stop=219, step=1) for column cast : unterminated string literal (detected at line 1) (<unknown>, line 1)
Error at row RangeIndex(start=598, stop=599, step=1) for column cast : unterminated string literal (detected at line 1) (<unknown>, line 1)
Error at row RangeIndex(start=637, stop=638, step=1) for column cast : unterminated string literal (detected at line 1) (<unknown>, line 1)
str
Error at row RangeIndex(start=28, stop=29, step=1) for column crew : unterminated string literal (detected at line 1) (<unknown>, line 1)
Error at row RangeIndex(start=212, stop=213, step=1) for column crew : '{' was never closed (<unknown>, line 1)
Error at row RangeIndex(start=0, stop=0, step=1) for column crew : malformed node or string: nan
Error at row RangeIndex(start=231, stop=232, step=1) for column crew : '{' was never closed (<unknown>, line 1)
Error at row RangeIndex(start=298, stop=299, step=1) for column crew : unterminated string li

In [47]:
print(df.head(5))

      budget                                         genres  \
0  237000000  [Action, Adventure, Fantasy, Science Fiction]   
1  300000000                   [Adventure, Fantasy, Action]   
2  245000000                     [Action, Adventure, Crime]   
3  250000000               [Action, Crime, Drama, Thriller]   
4  260000000           [Action, Adventure, Science Fiction]   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [culture clash, future, space war, space colon...                en   
1  [ocean, drug abuse, exotic island, east india ...                en   
2  [spy, based on novel, secret agent, seque

In [46]:
# now we need to manually fix the errors that were produced in the cell 2 above.
# these are just formatting errors within the csv
